# LSTM & GRU

**Companion lesson:** https://ml-viz.vercel.app/courses/rnns/03-lstm-and-gru

Run this notebook on Colab to experiment with the code from the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['figure.figsize'] = (8, 5)


## One LSTM step, fully spelled out

Three gates (forget, input, output) and an additive cell-state update.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

def lstm_step(x, h, c, p):
    z = np.concatenate([h, x])
    f = sigmoid(p['Wf'] @ z + p['bf'])
    i = sigmoid(p['Wi'] @ z + p['bi'])
    c_tilde = np.tanh(p['Wc'] @ z + p['bc'])
    o = sigmoid(p['Wo'] @ z + p['bo'])
    c = f * c + i * c_tilde   # additive update = gradient highway
    h = o * np.tanh(c)
    return h, c

np.random.seed(0)
H, X = 4, 3
p = {k: np.random.randn(H, H + X) * 0.1 for k in ['Wf','Wi','Wc','Wo']}
p.update({k: np.zeros(H) for k in ['bf','bi','bc','bo']})
p['bf'] += 1.0   # forget-gate bias trick: default to remembering
h, c = np.zeros(H), np.zeros(H)
for t in range(5):
    h, c = lstm_step(np.random.randn(X), h, c, p)
    print(f't={t}  cell={np.round(c, 2)}')

## PyTorch LSTM (if torch is installed)

In [ ]:
try:
    import torch, torch.nn as nn
    lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True)
    out, (h_n, c_n) = lstm(torch.randn(4, 15, 10))
    print('output', out.shape, '| h_n', h_n.shape, '| c_n', c_n.shape)
except ImportError:
    print('pip install torch to run this cell')